# Simuleringslab: fra molekyler til trykk og temperatur

## Del 1: Bli kjent med bridgechem

:::{admonition} Læringsmål
:class: note

Etter denne delen skal du kunne

* sette opp og kjøre en partikkelsimulering med `bridgechem`,
* hente ut rådata fra en simulering og forklare hva de inneholder,
* gjøre rede for hvorfor antall dimensjoner endrer tallene i termodynamikken,
* vurdere om en simulering er kjørt lenge nok og er fortynnet nok til å svare på spørsmålet du stiller.
:::

Du har lest at trykket i en gass kommer av at molekylene treffer veggene, at
temperaturen henger sammen med hvor fort de beveger seg, og at fartene
fordeler seg etter Maxwell og Boltzmann. Alt dette er påstander om noe vi ikke
kan se.

I denne labben skal vi se på det. `bridgechem` gir deg en boks med partikler
som følger Newtons lover og ingenting annet. Ingen gasslov er lagt inn, ingen
fartsfordeling er antatt. Alt vi skal måle må komme ut av bevegelsen selv.

Del 1 handler bare om å bli kjent med verktøyet. De faktiske målingene kommer
i del 2, og rapporten skriver du i malen til slutt.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import bridgechem as bc

from bridgechem.constants import K_B, N_A, gas_properties

plt.rcParams.update({"figure.dpi": 110, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

M_ARGON = gas_properties("argon")["mass_kg"]
print(f"Massen til ett argonatom: {M_ARGON:.3e} kg")

## 1.1 Den første boksen

En boks lages med `bc.box`. Du sier hvor mange partikler du vil ha og hvor
stor boksen skal være, i nanometer.

:::{admonition} Før du kjører: hva tror du?
:class: tip

Vi lager en kube på 12 nm med 120 argonatomer ved 300 K. Argonatomene har en
diameter på rundt 0,3 nm. Tegn for deg selv hvordan du tror det ser ut. Er
boksen full eller nesten tom? Beveger partiklene seg langsomt nok til at du
rekker å følge én av dem?
:::

In [ ]:
system = bc.box(N=120, size=(12, 12, 12), temperature=300, seed=0)
print(system)

sim = system.run(t=400)   # 400 pikosekunder

Tre tall i `size` gir en tredimensjonal boks. To tall gir en flat boks. Det
valget får konsekvenser lenger ned, og vi kommer tilbake til det i 1.3.

En 3D-boks tegnes som sin skygge i xy-planet. To partikler kan derfor se ut
som om de kolliderer selv om de passerer hverandre med god margin i
z-retning. `show(slab=3)` viser bare en 3 nm tykk skive gjennom midten, og da
er det du ser faktisk det som skjer.

In [ ]:
sim.show(slab=3.0, vectors=True)

:::{admonition} Underveisoppgave 1: temperatur og fart
:class: tip

Kjør den samme boksen ved 100 K og ved 900 K. Se på simuleringen begge
ganger.

1. Hvor mye raskere beveger partiklene seg ved 900 K enn ved 100 K? Gjett
   først, se etterpå.
2. Regn ut forholdet mellom rms-farten ved de to temperaturene med
   `bc.rms_speed(T, M_ARGON, dim=3)`. Stemmer det med det du så?
:::

In [ ]:
# Din kode her

:::{admonition} Løsningsforslag
:class: dropdown

```python
for T in (100, 900):
    print(f"T = {T:4d} K:  v_rms = {bc.rms_speed(T, M_ARGON, dim=3):6.1f} m/s")
```

Rms-farten går som kvadratroten av temperaturen, så en nidobling av $T$ gir
bare en tredobling av farten:

$$v_\mathrm{rms} = \sqrt{\frac{3k_\mathrm{B}T}{m}}$$

Det er lett å undervurdere. Ni ganger så varmt ser ikke ut som ni ganger så
travelt, og det er nettopp fordi det er energien, ikke farten, som er
proporsjonal med $T$.
:::

## 1.2 Hva ligger i en `Simulation`?

`run()` gir deg tilbake et `Simulation`-objekt. Det inneholder hele banen til
alle partiklene, ikke bare et sluttresultat. Det er dette som gjør at vi kan
måle ting i etterkant i stedet for å be simuleringen om et fasitsvar.

In [ ]:
posisjoner = sim.calculate("velocities")   # (n_frames, N, dim), m/s
farter = sim.calculate("speeds")           # (n_frames, N), m/s

print("hastigheter:", sim.calculate("velocities").shape)
print("farter:     ", farter.shape)
print("tider:      ", sim.times.shape, f"  siste tid: {sim.total_time*1e12:.1f} ps")
print("volum:      ", f"{sim.volume:.3e} m^3")
print("trykkenhet: ", sim.pressure_unit)

:::{admonition} Underveisoppgave 2: les av formene
:class: tip

`sim.calculate("velocities")` har tre akser.

1. Hva står hver av de tre aksene for?
2. Hvor mange tall er det til sammen i den matrisen?
3. `sim.times` har bare én akse. Hvorfor?
:::

:::{admonition} Løsningsforslag
:class: dropdown

Aksene er `(bilde, partikkel, komponent)`. Første akse er hvilket øyeblikk i
simuleringen vi ser på, andre akse er hvilken partikkel, og tredje akse er
$x$, $y$ og $z$ av hastighetsvektoren til den partikkelen.

Antall tall er produktet av de tre, altså `n_frames * N * 3`.

`sim.times` har én akse fordi tiden er felles for alle partiklene. Det er ett
klokkeslett per bilde, ikke ett per partikkel.

Legg merke til at simuleringen lagrer *hastigheter*, altså vektorer, mens vi
oftest regner med *farter*, altså lengden av dem. Det er derfor `speeds` har
én akse mindre.
:::

## 1.3 To eller tre dimensjoner

Antall dimensjoner er ikke bare en tegneteknisk detalj. Det står midt i
ekvipartisjonsprinsippet.

:::{admonition} Ekvipartisjonsprinsippet
:class: important

Hvert kvadratiske ledd i energien til en partikkel bidrar med
$\tfrac{1}{2}k_\mathrm{B}T$ i gjennomsnitt ved likevekt.

En fri partikkel har ett slikt ledd per romretning, siden
$E_\mathrm{kin} = \tfrac{1}{2}m(v_x^2 + v_y^2 + v_z^2)$. Den gjennomsnittlige
kinetiske energien blir derfor

$$\langle E_\mathrm{kin} \rangle = \frac{d}{2} k_\mathrm{B} T$$

der $d$ er antall dimensjoner.
:::

:::{admonition} Før du kjører: hva tror du?
:class: tip

Vi lager to bokser med samme antall partikler og samme temperatur, én flat og
én i tre dimensjoner. Blir den gjennomsnittlige kinetiske energien per
partikkel den samme i de to?
:::

In [ ]:
for dim, size in ((2, (40, 40)), (3, (12, 12, 12))):
    s = bc.box(N=120, size=size, temperature=300, radius=0.08, seed=0)
    r = s.run(t=200, animate=False)
    T = float(np.mean(r.calculate("temperature")))
    E_kin = np.mean(r.calculate("kinetic_energy")) / r.n_particles
    print(f"{dim}D:  T = {T:6.1f} K    <E_kin> per partikkel = {E_kin/(K_B*T):.3f} kT")

:::{admonition} Underveisoppgave 3: hvor blir det av den tredje halvdelen?
:class: tip

1. Forklar tallene du fikk ut, med utgangspunkt i ekvipartisjonsprinsippet.
2. Fartsfordelingen ser også forskjellig ut i 2D og 3D. Plott begge med
   `bc.maxwell_boltzmann_speed(v, 300, M_ARGON, dim=2)` og `dim=3` i samme
   figur. Hvilken har toppen lengst til høyre, og hvorfor?
3. Vi kommer til å bruke 3D i resten av labben. Nevn én grunn som handler om
   fysikk, og én som handler om enheter.
:::

In [ ]:
# Din kode her

:::{admonition} Løsningsforslag
:class: dropdown

**1.** I 2D har partikkelen to kvadratiske ledd i energien, i 3D har den tre.
Ved samme temperatur er derfor $\langle E_\mathrm{kin}\rangle = k_\mathrm{B}T$
i 2D og $\tfrac{3}{2}k_\mathrm{B}T$ i 3D. Temperaturen er den samme fordi
temperatur er *energi per frihetsgrad*, ikke energi per partikkel.

**2.**

```python
v = np.linspace(0, 1200, 400)
for dim in (2, 3):
    plt.plot(v, bc.maxwell_boltzmann_speed(v, 300, M_ARGON, dim=dim),
             label=f"{dim}D")
plt.xlabel("fart (m/s)"); plt.ylabel("sannsynlighetstetthet"); plt.legend();
```

3D-kurven ligger lengst til høyre. En partikkel i 3D har én ekstra retning å
lagre kinetisk energi i, så ved samme $T$ er den totale farten større:
$v_\mathrm{rms} = \sqrt{3k_\mathrm{B}T/m}$ mot $\sqrt{2k_\mathrm{B}T/m}$.

**3.** Fysikken: en ekte gass er tredimensjonal, og faktoren $\tfrac{3}{2}$ er
den som står i læreboka. Enhetene: i 3D er trykk kraft per areal og måles i
pascal, som du kan sammenlikne med 1 bar. I 2D finnes det ikke noe areal å
trykke mot, bare linjene rundt boksen, så trykk blir kraft per lengde i N/m.
Det er en størrelse du aldri har møtt.
:::

## 1.4 Størrelsesorden: hvor mye gass er dette egentlig?

Boksene våre er noen få nanometer på hver kant. Det er verdt å vite hva det
svarer til før vi begynner å måle.

In [ ]:
side_nm = 18.4
V = (side_nm * 1e-9) ** 3
T = 300.0

# Hvor mange partikler trengs for 1 bar i denne boksen?
N_1bar = 1.013e5 * V / (K_B * T)
print(f"Kube med side {side_nm} nm gir V = {V:.3e} m^3 = {V*1e24:.1f} nm^3")
print(f"For 1 bar ved {T:.0f} K trengs N = {N_1bar:.0f} partikler")
print(f"Det svarer til n = {N_1bar/N_A:.3e} mol")

:::{admonition} Underveisoppgave 4: fra nanometer til laboratoriet
:class: tip

1. Hvor mange argonatomer er det i 1 cm$^3$ ved 1 bar og 300 K?
2. Simuleringen bruker rundt et sekund per 100 partikler for en kort kjøring.
   Anslå hvor lang tid det ville tatt å simulere den kubikkcentimeteren.
3. Vi skal likevel bruke resultatene fra 150 partikler til å si noe om ekte
   gasser. Hvorfor er ikke det juks?
:::

:::{admonition} Løsningsforslag
:class: dropdown

**1.** $N = pV/k_\mathrm{B}T = (1{,}013\cdot 10^5 \cdot 10^{-6}) /
(1{,}381\cdot10^{-23}\cdot 300) \approx 2{,}4\cdot 10^{19}$ atomer.

**2.** Det er over $10^{17}$ ganger flere partikler enn i boksen vår, og
regnetiden vokser omtrent som $N^2$ fordi vi sjekker alle par. Svaret blir et
tall uten mening, langt lengre enn universets alder. Poenget er ikke det
eksakte tallet, men at brute force aldri kommer til å være veien.

**3.** Fordi trykk og temperatur er *intensive* størrelser. De avhenger av
tetthet og energi per partikkel, ikke av hvor mange partikler du har. Det vi
mister med få partikler er ikke riktigheten av middelverdien, men roen i den:
fluktuasjonene rundt middelverdien går som $1/\sqrt{N}$. Med 150 partikler er
de fortsatt små nok til at vi får tre siffer på trykket, og det skal vi vise
i del 2.
:::

## 1.5 Rådata: hva simuleringen faktisk måler

Dette er det viktigste avsnittet i del 1.

`bridgechem` kan regne ut trykket for deg. Men da har du ikke målt noe, du har
bare kalt en funksjon. Derfor gir simuleringen deg også råmaterialet:
`sim.wall_collisions()` er den samlede bevegelsesmengden partiklene har
overført til veggene, akse for akse.

Hver gang en partikkel med hastighetskomponent $v_x$ spretter av en vegg som
står vinkelrett på $x$, snur $v_x$ fortegn. Bevegelsesmengden partikkelen
mister, og veggen mottar, er $2m|v_x|$. Summerer vi det over alle støt i et
tidsrom $\Delta t$ og deler på tiden og på veggarealet, får vi en kraft per
areal. Altså et trykk.

In [ ]:
system = bc.box(N=150, size=(18.4, 18.4, 18.4), temperature=300,
                radius=0.08, seed=0)
sim = system.run(t=2000, animate=False)

impuls = sim.wall_collisions()          # (3,) i kg m/s, én per akse
veggareal = sim.volume / sim.L          # (3,) arealet av hver vegg

print("overført bevegelsesmengde per akse (kg m/s):", impuls)
print("veggareal per akse (m^2):                   ", veggareal)

# Regn trykket for hånd
P_for_hand = np.mean(impuls / (sim.total_time * 2 * veggareal))
print(f"\nTrykk regnet for hånd:   {P_for_hand:10.1f} Pa")
print(f"Trykk fra biblioteket:   {sim.pressure('wall'):10.1f} Pa")

Faktoren 2 i nevneren er fordi hver akse har to vegger, én i hver ende.

Trykket finnes også som en tidsserie, slik at du kan se om simuleringen har
roet seg.

In [ ]:
P_t = sim.pressure("wall", per_frame=True)

plt.figure(figsize=(7, 3.2))
plt.plot(sim.times * 1e12, P_t / 1e5, lw=0.8, color="#2b6cb0")
plt.axhline(sim.pressure("wall") / 1e5, color="#c05621", lw=2,
            label="gjennomsnitt over hele kjøringen")
plt.xlabel("tid (ps)"); plt.ylabel("trykk (bar)"); plt.legend()
plt.title("Trykket svinger fra bilde til bilde, men middelverdien ligger stille");

:::{admonition} Underveisoppgave 5: hvorfor svinger det så mye?
:class: tip

1. Enkeltbildene spriker kraftig, men middelverdien er stabil. Forklar
   hvorfor, med utgangspunkt i hvor mange veggstøt som rekker å skje i løpet
   av ett bilde.
2. Bruk `np.std(P_t) / np.mean(P_t)` til å tallfeste spredningen. Kjør så det
   samme med `N=600` i en boks med fire ganger volumet, altså samme tetthet.
   Hvordan endrer den relative spredningen seg?
:::

In [ ]:
# Din kode her

:::{admonition} Løsningsforslag
:class: dropdown

**1.** Trykket i ett enkelt bilde er summen av noen få tilfeldige støt.
Kommer det tilfeldigvis to raske partikler mot veggen i det bildet, spretter
tallet opp. Over hele kjøringen summeres titusener av støt, og de tilfeldige
utslagene jevner hverandre ut.

**2.**

```python
for N, side in ((150, 18.4), (600, 29.2)):
    s = bc.box(N=N, size=(side,)*3, temperature=300, radius=0.08, seed=0)
    r = s.run(t=800, animate=False)
    P = r.pressure("wall", per_frame=True)[1:]
    print(f"N = {N:3d}:  relativ spredning = {np.std(P)/np.mean(P):.3f}")
```

Den relative spredningen synker omtrent som $1/\sqrt{N}$. Firedobler du
antallet partikler, halveres støyen. Det er den samme statistikken som gjør
at makroskopiske gasser har helt stabilt trykk: med $10^{19}$ partikler er
fluktuasjonene i tiende desimal.
:::

## 1.6 To feller du bør kjenne til før du måler

### Felle 1: standardboksen er ingen ideell gass

Når du ikke oppgir `radius`, velger `bridgechem` partikler som er store nok
til at du ser dem, nærmere bestemt slik at de fyller ti prosent av boksen. Det
er praktisk å se på, men ti prosent fylling er *ikke* en fortynnet gass.

In [ ]:
for merke, radius in (("standard (synlig)", None), ("fortynnet", 0.08)):
    s = bc.box(N=150, size=(18.4, 18.4, 18.4), temperature=300,
               radius=radius, seed=0)
    r = s.run(t=1500, animate=False)
    forhold = r.pressure("virial") / r.ideal_gas_pressure()
    print(f"{merke:<18} r = {s.radius[0]*1e9:.3f} nm    P/P_ideell = {forhold:.3f}")

Avviket er ekte fysikk. Partikler med utstrekning stenger hverandre ute fra en
del av volumet, så den frie plassen er mindre enn $V$, og trykket blir høyere
enn $Nk_\mathrm{B}T/V$. Det er nettopp $b$-leddet i van der Waals-likningen.
Men vil du sammenlikne med den ideelle gassloven, må du gjøre partiklene små.

Du kan fortsatt se dem: `display_scale` endrer bare tegningen, ikke fysikken.

### Felle 2: `steps` er ikke tid

Tidsskrittet `dt` velges automatisk, og det krymper når partiklene blir mindre
eller raskere. Et fast antall `steps` dekker derfor *kortere fysisk tid* for
små partikler. Bruk `t` i pikosekunder når varigheten betyr noe, og det gjør
den for enhver måling.

In [ ]:
print("Samme antall steg, to partikkelstørrelser:")
for radius in (0.4, 0.05):
    r = bc.box(N=100, size=(18.4,)*3, radius=radius, seed=0).run(steps=20000, animate=False)
    print(f"  r = {radius:.2f} nm  ->  {r.total_time*1e12:7.1f} ps simulert")

print("\nSamme varighet, to partikkelstørrelser:")
for radius in (0.4, 0.05):
    r = bc.box(N=100, size=(18.4,)*3, radius=radius, seed=0).run(t=500, animate=False)
    print(f"  r = {radius:.2f} nm  ->  {r.total_time*1e12:7.1f} ps simulert")

:::{admonition} Underveisoppgave 6: hvor lenge er lenge nok?
:class: tip

Mål trykket i den fortynnede boksen (`radius=0.08`, `N=150`, 18,4 nm kube) for
`t = 100, 300, 1000, 3000` ps. Sammenlikn hver gang med
`ideal_gas_pressure()`.

1. Hvor lenge må du kjøre for å komme innenfor to prosent?
2. Gjenta med tre forskjellige `seed`. Hvor mye varierer svaret mellom dem ved
   den korteste og den lengste kjøretiden?
3. Formuler en regel du kan bruke i del 2.
:::

In [ ]:
# Din kode her

:::{admonition} Løsningsforslag
:class: dropdown

```python
for t in (100, 300, 1000, 3000):
    verdier = []
    for seed in range(3):
        r = bc.box(N=150, size=(18.4,)*3, temperature=300,
                   radius=0.08, seed=seed).run(t=t, animate=False)
        verdier.append(r.pressure("wall") / r.ideal_gas_pressure())
    verdier = np.array(verdier)
    print(f"t = {t:5d} ps:  P/P_ideell = {verdier.mean():.3f} "
          f"+- {verdier.std():.3f}")
```

Ved 100 ps spriker svarene med flere prosent mellom seedene, og du kan lett
"måle" et avvik fra gassloven som bare er støy. Ved 1000 til 3000 ps er
spredningen nede i noen få promille.

Regelen: kjør minst 1000 ps for en trykkmåling, og sjekk alltid at
gjentakelser med ulik `seed` gir samme svar innenfor den nøyaktigheten du
trenger. Et enkelt tall uten en anelse om spredningen er ikke en måling.
:::

## Sjekkliste før del 2

Du er klar når du kan svare på dette uten å slå opp:

* Hvordan lager du en tredimensjonal boks med gitt antall partikler, størrelse
  og temperatur?
* Hvor henter du ut rå hastigheter, og hva betyr de tre aksene i den matrisen?
* Hva er $\langle E_\mathrm{kin}\rangle$ per partikkel i 3D, uttrykt ved
  $k_\mathrm{B}T$?
* Hvorfor må du sette en liten `radius` når du vil sammenlikne med den ideelle
  gassloven?
* Hvorfor bruker vi `t` og ikke `steps`?

I del 2 skal du måle fartsfordelingen, temperaturen, trykket og den indre
energien, og sammenlikne hver av dem med det teorien forutsier.